In [22]:
# ===== 1) IMPORTS & GLOBAL CONFIG =====
from __future__ import annotations

import re
import time
import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ---- What to scrape (edit these) ----
SCHOOLS = {
    # Use {year} in each URL; keep ?view=2 for the table/grid view when available.
    "Jackson State": "https://gojsutigers.com/sports/football/roster/{year}?view=2",
    "Alabama State": "https://bamastatesports.com/sports/football/roster/{year}?view=2",
    "Alabama A&M": "https://aamusports.com/sports/football/roster/2025{year}?view=2",
    "Southern": "https://gojagsports.com/sports/football/roster/{year}?view=2",
    "Prairie View A&M": "https://pvpanthers.com/sports/football/roster/{year}?view=2",
    "Texas Southern": "https://tsusports.com/sports/football/roster/{year}?view=2",
    "UAPB": "https://uapblionsroar.com/sports/football/roster/{year}?view=2",
    "Alcorn State": "https://alcornsports.com/sports/football/roster/{year}?view=2",
    "Grambling": "https://gsutigers.com/sports/football/roster/{year}?view=2",
    "Mississippi Valley State": "https://mvsusports.com/sports/football/roster/{year}?view=2",
    "Florida A&M": "https://famuathletics.com/sports/football/roster/{year}?view=2",
    "Bethune-Cookman": "https://bcuathletics.com/sports/football/roster/{year}?view=2"

}

YEARS = list(range(2010, 2026))     # inclusive range you want to scrape
HEADLESS = True                     # set False to watch the browser work
PAGE_LOAD_TIMEOUT = 20              # seconds to wait for page/table
POLITE_DELAY = 1.5                  # seconds between requests (be nice)
EMPTY_ROW_THRESHOLD = 1              # 0–1 row = “empty”
AUTO_STOP_CONSECUTIVE = 4            # stop after 4 empty years in a row

In [23]:
# ===== 2) SELENIUM DRIVER FACTORY =====
def make_driver(headless: bool = True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    # Selenium Manager will fetch the right ChromeDriver automatically.
    driver = webdriver.Chrome(options=opts)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return driver


In [24]:
# ===== 3) SCRAPER (ONE PAGE) =====
def scrape_roster_table(driver, url: str) -> pd.DataFrame | None:
    """Load Sidearm roster ?view=2 table and return a DataFrame or None."""
    try:
        driver.get(url)
    except Exception as e:
        print(f"[!] Page load failed: {url}\n    {e}")
        return None

    try:
        WebDriverWait(driver, PAGE_LOAD_TIMEOUT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "table.sidearm-table"))
        )
        time.sleep(0.75)
    except Exception:
        print(f"[!] No roster table detected: {url}")
        return None

    soup = BeautifulSoup(driver.page_source, "lxml")
    table = soup.find("table", class_="sidearm-table")
    if not table:
        return None

    thead = table.find("thead")
    headers = [th.get_text(" ", strip=True) for th in thead.find_all("th")] if thead else []

    tbody = table.find("tbody")
    if not tbody:
        return None

    rows = []
    for tr in tbody.find_all("tr"):
        tds = tr.find_all("td")
        if not tds:
            continue
        rows.append([td.get_text(" ", strip=True) for td in tds])

    if not rows:
        return None

    df = pd.DataFrame(rows)
    if len(headers) == df.shape[1]:
        df.columns = headers
    return df


In [26]:
# ===== 4) NORMALIZATION HELPERS (fixed for dup columns + concat error) =====
HEADER_MAP = {
    "full name": "name", "name": "name",
    "pos.": "pos", "position": "pos",
    "ht.": "ht", "height": "ht",
    "wt.": "wt", "weight": "wt",
    "yr.": "yr", "year": "yr",
    "#": "number", "no.": "number",
    "image": "image",
    # Combined columns used by various Sidearm sites:
    "hometown/previous school": "home_prev",
    "hometown / previous school": "home_prev",
    "hometown/high school": "home_prev",
    "hometown / high school": "home_prev",
}

HS_PATTERNS = re.compile(
    r"(?:\bHS\b|\bHigh School\b|\bPrep\b|\bAcademy\b|\bSchool\b)$",
    re.IGNORECASE,
)

def canonicalize_headers(df: pd.DataFrame) -> pd.DataFrame:
    """Map different spellings to consistent names. Do NOT drop dups here."""
    df = df.copy()
    mapped = []
    for c in df.columns:
        key = re.sub(r"\s+", " ", c.strip().lower())
        mapped.append(HEADER_MAP.get(key, c))
    df.columns = mapped
    return df

def split_hometown_prev(s: str):
    """Return (hometown, state, high_school, previous_schools) from 'City, ST / School...'."""
    if not isinstance(s, str) or not s.strip():
        return None, None, None, None

    parts = [p.strip(" -\u2013\u2014").strip() for p in s.split("/") if p.strip()]
    city_state = parts[0] if parts else ""
    hometown, state = None, None
    if "," in city_state:
        left, right = city_state.split(",", 1)
        hometown = left.strip()
        state = right.strip()
    else:
        hometown = city_state.strip()

    schools = parts[1:] if len(parts) > 1 else []
    high_school, previous_schools = None, None
    if schools:
        last = schools[-1]
        if HS_PATTERNS.search(last):
            high_school = last
            prior = schools[:-1]
        else:
            prior = schools
        if prior:
            previous_schools = " / ".join(prior)

    return hometown or None, state or None, high_school, previous_schools

def expand_home_prev(df: pd.DataFrame) -> pd.DataFrame:
    """
    Safely expand combined column into separate parts.
    Handles duplicate 'home_prev' columns by coalescing row-wise.
    """
    if "home_prev" not in df.columns:
        return df

    out = df.copy()

    # All columns that are exactly 'home_prev' (could be duplicates after mapping)
    hp_mask = out.columns == "home_prev"
    hp_cols = list(out.columns[hp_mask])

    if len(hp_cols) == 1:
        hp_series = out["home_prev"]
    else:
        # Coalesce across duplicates: take first non-empty cell per row
        hp_df = out.loc[:, hp_mask]
        def pick_first_nonempty(row):
            for val in row:
                if pd.notna(val) and str(val).strip():
                    return str(val).strip()
            return ""
        hp_series = hp_df.apply(pick_first_nonempty, axis=1)

        # Drop extras, keep a single logical 'home_prev'
        to_drop = hp_cols[1:]
        out = out.drop(columns=to_drop)
        out.loc[:, "home_prev"] = hp_series

    hp_clean = hp_series.where(hp_series.notna(), "")
    parsed = hp_clean.map(split_hometown_prev)

    # Coerce anything odd to a 4-tuple
    def _coerce4(x):
        if isinstance(x, (list, tuple)) and len(x) == 4:
            return tuple(x)
        try:
            lst = list(x)
            return tuple(lst[:4]) + (None,) * (4 - len(lst))
        except Exception:
            return (None, None, None, None)

    parts = [_coerce4(p) for p in parsed]

    parts_df = pd.DataFrame(
        parts,
        index=out.index,
        columns=["hometown", "state", "high_school", "previous_schools"],
    )

    out = pd.concat([out, parts_df], axis=1)
    return out

def normalize_roster(df: pd.DataFrame) -> pd.DataFrame:
    """
    Unify column names, expand combined hometown/prev-school, ensure unique labels,
    and return a consistent schema for concat.
    """
    df = canonicalize_headers(df).copy()

    # If an explicit HS column exists, keep it
    hs_like = [c for c in df.columns if c.strip().lower() in {"high school", "hs"}]
    if hs_like:
        df.loc[:, "high_school"] = df[hs_like[0]]

    # Expand combined column (handles duplicate 'home_prev' internally)
    df = expand_home_prev(df)

    # Ensure unique column labels BEFORE selecting final columns
    df = df.loc[:, ~df.columns.duplicated()].copy()

    # Ensure high_school exists
    if "high_school" not in df.columns:
        df["high_school"] = pd.NA

    # Final, consistent schema
    preferred_cols = [
        "number", "name", "pos", "ht", "wt", "yr",
        "hometown", "state", "high_school", "previous_schools",
    ]
    for c in preferred_cols:
        if c not in df.columns:
            df[c] = pd.NA

    return df[preferred_cols].copy()




In [27]:
# ===== 5) MAIN LOOP: DESC + AUTO-STOP =====
def scrape_schools_years_desc_autostop(schools: dict[str,str], years: list[int]) -> pd.DataFrame:
    driver = make_driver(HEADLESS)
    all_frames = []
    try:
        for school, template in schools.items():
            print(f"\n=== {school} ===")
            school_frames = []
            consecutive_empty = 0

            for yr in sorted(years, reverse=True):  # newest → oldest
                url = template.format(year=yr)
                print(f"  Year {yr}: {url}")

                df = scrape_roster_table(driver, url)
                if df is None or df.shape[0] <= EMPTY_ROW_THRESHOLD:
                    consecutive_empty += 1
                    print(f"    -> Skipped (empty or <= {EMPTY_ROW_THRESHOLD} row). "
                          f"[{consecutive_empty} empty in a row]")
                else:
                    consecutive_empty = 0
                    norm = normalize_roster(df).copy()
                    norm.loc[:, "school"] = school
                    norm.loc[:, "year"] = yr
                    school_frames.append(norm)
                    print(f"    -> Got {norm.shape[0]} rows.")

                # Auto-stop once we’ve seen many empty years consecutively
                if consecutive_empty >= AUTO_STOP_CONSECUTIVE:
                    print(f"    -> Auto-stop for {school} "
                          f"(no real data last {AUTO_STOP_CONSECUTIVE} years).")
                    break

                time.sleep(POLITE_DELAY)

            if school_frames:
                school_df = pd.concat(school_frames, ignore_index=True, sort=False)
                out_name = f"{school.replace(' ','_').replace('&','and').lower()}_rosters.csv"
                school_df.to_csv(out_name, index=False)
                print(f"  Saved per-school CSV: {out_name}")
                all_frames.append(school_df)
            else:
                print("  (No rows for this school.)")

    finally:
        driver.quit()

    if not all_frames:
        print("[!] Nothing scraped — check URLs and years.")
        return pd.DataFrame()
    return pd.concat(all_frames, ignore_index=True, sort=False)



In [28]:
# ===== 6) RUN =====
master_df = scrape_schools_years_desc_autostop(SCHOOLS, YEARS)

print("\n--- SUMMARY ---")
print(master_df.shape)
if not master_df.empty:
    print(master_df.groupby(["school","year"]).size().sort_index())
    master_df.to_csv("all_rosters.csv", index=False)
    print("\nSaved master: all_rosters.csv")
else:
    print("No data scraped.")




=== Jackson State ===
  Year 2025: https://gojsutigers.com/sports/football/roster/2025?view=2
    -> Got 96 rows.
  Year 2024: https://gojsutigers.com/sports/football/roster/2024?view=2
    -> Got 101 rows.
  Year 2023: https://gojsutigers.com/sports/football/roster/2023?view=2
    -> Got 105 rows.
  Year 2022: https://gojsutigers.com/sports/football/roster/2022?view=2
    -> Got 95 rows.
  Year 2021: https://gojsutigers.com/sports/football/roster/2021?view=2
    -> Skipped (empty or <= 1 row). [1 empty in a row]
  Year 2020: https://gojsutigers.com/sports/football/roster/2020?view=2
    -> Skipped (empty or <= 1 row). [2 empty in a row]
  Year 2019: https://gojsutigers.com/sports/football/roster/2019?view=2
    -> Got 83 rows.
  Year 2018: https://gojsutigers.com/sports/football/roster/2018?view=2
    -> Got 90 rows.
  Year 2017: https://gojsutigers.com/sports/football/roster/2017?view=2
    -> Skipped (empty or <= 1 row). [1 empty in a row]
  Year 2016: https://gojsutigers.com/sport

In [ ]:
master_df





NameError: name 'master_df' is not defined